# Notebook 06 — End-to-End Pipeline Evaluation
Simulate the full WAF pipeline (L1 → L2A → L2B → Threat Score) on the test set. Measure overall detection rate, FPR, and latency.

This produces the final numbers for your project report.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort
import time, re, json
from pathlib import Path

from feature_engineering.extractor import extract_features, to_vector, FEATURE_NAMES
from feature_engineering.tokenizer import CharTokenizer
from feature_engineering.normalizer import FeatureNormalizer

SPLITS  = Path("../data/splits")
EXPORTS = Path("../exported_models")
DATA_PROC = Path("../data/processed")

CLASS_NAMES = ["normal", "sqli", "xss", "lfi", "other_attack"]

## 1. Load ONNX models and normalizer

In [ ]:
l2a_sess  = ort.InferenceSession(str(EXPORTS / "layer2a_best.onnx"))
l2b_sess  = ort.InferenceSession(str(EXPORTS / "layer2b_best.onnx"))
norm      = FeatureNormalizer.load(str(EXPORTS / "scaler_l2a.pkl"))
tok       = CharTokenizer(max_len=512)

# load winner names
with open(EXPORTS / "l2a_results.json") as f: l2a_winner = json.load(f)["winner"]
with open(EXPORTS / "l2b_results.json") as f: l2b_winner = json.load(f)["winner"]

# load threshold for L2A
thr_path = str(EXPORTS / "layer2a_best_threshold.txt")
with open(thr_path) as f: L2A_THRESHOLD = float(f.read().strip())

l2a_input_name = l2a_sess.get_inputs()[0].name
l2b_input_name = l2b_sess.get_inputs()[0].name
l2b_uses_tokens = (l2b_input_name == "token_ids")

print(f"L2A model: {l2a_winner}  input='{l2a_input_name}'  threshold={L2A_THRESHOLD:.5f}")
print(f"L2B model: {l2b_winner}  input='{l2b_input_name}'")

## 2. Layer 1 rule engine (inline)

In [ ]:
SQLI_RE = re.compile(r"union.*select|or \d+=\d+|drop table|--.*select", re.I)
XSS_RE  = re.compile(r"<script|onerror|javascript:|alert\(", re.I)
LFI_RE  = re.compile(r"\.\./|/etc/passwd|boot\.ini", re.I)

def layer1_check(url: str, body: str) -> tuple:
    """Returns (blocked: bool, reason: str)"""
    txt = url + " " + body
    if SQLI_RE.search(txt): return True, "sqli_rule"
    if XSS_RE.search(txt):  return True, "xss_rule"
    if LFI_RE.search(txt):  return True, "lfi_rule"
    return False, ""

print("Layer 1 rules loaded.")

## 3. Full pipeline function

In [ ]:
def run_pipeline(request: dict) -> dict:
    """
    Run a single request through the full WAF pipeline.
    Returns a result dict with all intermediate decisions.
    """
    url  = str(request.get("url", ""))
    body = str(request.get("body", ""))
    t0   = time.perf_counter()

    # Layer 1
    l1_blocked, l1_reason = layer1_check(url, body)
    if l1_blocked:
        return {"layer": "L1", "decision": "block", "score": 100,
                "label": l1_reason, "latency_ms": (time.perf_counter()-t0)*1000}

    # Layer 2A
    features = extract_features(request)
    fvec     = norm.transform(to_vector(features))   # (1, 20)

    if l2a_winner == "shallow_autoencoder":
        recon = l2a_sess.run(None, {l2a_input_name: fvec})[0]
        l2a_score = float(np.mean((fvec - recon) ** 2))
    else:
        # isolation forest: decision_function output, negated
        raw = l2a_sess.run(None, {l2a_input_name: fvec})
        l2a_score = float(-raw[0][0]) if hasattr(raw[0], '__len__') else float(-raw[0])

    l2a_is_anomaly = l2a_score >= L2A_THRESHOLD

    if not l2a_is_anomaly:
        return {"layer": "L2A", "decision": "allow", "score": 0,
                "label": "normal", "latency_ms": (time.perf_counter()-t0)*1000,
                "l2a_score": round(l2a_score, 5)}

    # Layer 2B
    if l2b_uses_tokens:
        tokens = tok.encode_request(request).reshape(1, -1).astype(np.int32)
        logits = l2b_sess.run(None, {l2b_input_name: tokens})[0][0]
    else:
        logits = l2b_sess.run(None, {l2b_input_name: fvec})[0][0]

    import scipy.special
    proba     = scipy.special.softmax(logits)
    pred_cls  = int(np.argmax(proba))
    pred_conf = float(proba[pred_cls])
    pred_label = CLASS_NAMES[pred_cls]

    # Threat score: L2A anomaly + L2B confidence
    l2a_contrib = min(50.0, l2a_score * 15)
    l2b_contrib = 0.0 if pred_cls == 0 else pred_conf * 50
    threat_score = min(100, int(l2a_contrib + l2b_contrib))

    if threat_score >= 70:   decision = "block"
    elif threat_score >= 30: decision = "log"
    else:                    decision = "allow"

    return {
        "layer": "L2B", "decision": decision,
        "score": threat_score, "label": pred_label,
        "confidence": round(pred_conf, 4),
        "l2a_score": round(l2a_score, 5),
        "l2a_contrib": round(l2a_contrib, 2),
        "l2b_contrib": round(l2b_contrib, 2),
        "latency_ms": round((time.perf_counter()-t0)*1000, 3),
    }

print("Pipeline function ready.")

## 4. Run pipeline over full test set

In [ ]:
df = pd.read_csv(DATA_PROC / "csic_parsed.csv")
X_te_num  = np.load(SPLITS / "l2b_test_X_numeric.npy")
X_te_tok  = np.load(SPLITS / "l2b_test_X_tokens.npy")
y_te      = np.load(SPLITS / "l2b_test_y.npy")
te_indices = np.load(SPLITS / "l2b_test_y.npy")  # same length as test split

# Re-build test requests from df using the same test split indices
from sklearn.model_selection import train_test_split
idx_all = np.arange(len(df))
y_mc    = df["attack_class_id"].values
_, idx_temp = train_test_split(idx_all, test_size=0.30, stratify=y_mc, random_state=42)
_, idx_te   = train_test_split(idx_temp, test_size=0.50,
                               stratify=y_mc[idx_temp], random_state=42)

test_df = df.iloc[idx_te].reset_index(drop=True)

print(f"Running pipeline on {len(test_df)} test requests...")
results_list = []
for i, row in test_df.iterrows():
    req = {"url": str(row.get("url","")), "method": str(row.get("method","GET")),
           "headers": {}, "body": str(row.get("body",""))}
    res = run_pipeline(req)
    res["true_label"] = row["attack_class"]
    res["true_label_id"] = row["attack_class_id"]
    results_list.append(res)

df_results = pd.DataFrame(results_list)
print(df_results["decision"].value_counts())

## 5. Overall detection metrics

In [ ]:
y_true   = (df_results["true_label_id"] > 0).astype(int)
y_pred   = (df_results["decision"] == "block").astype(int)

from sklearn.metrics import confusion_matrix, classification_report
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
fpr  = fp / (fp + tn)
tpr  = tp / (tp + fn)
prec = tp / (tp + fp) if (tp+fp) > 0 else 0
acc  = (tp + tn) / len(y_true)

print("=== End-to-End Pipeline Results ===")
print(f"  Accuracy:          {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Detection rate:    {tpr:.4f}  ({tpr*100:.2f}%)")
print(f"  False positive rate: {fpr:.4f}  ({fpr*100:.2f}%)")
print(f"  Precision:         {prec:.4f}")
print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")

## 6. Latency analysis

In [ ]:
lat = df_results["latency_ms"]
print("=== Latency (full pipeline, ms) ===")
print(f"  Mean:  {lat.mean():.3f}")
print(f"  P50:   {lat.quantile(0.50):.3f}")
print(f"  P95:   {lat.quantile(0.95):.3f}")
print(f"  P99:   {lat.quantile(0.99):.3f}")
print(f"  Max:   {lat.max():.3f}")

plt.figure(figsize=(8, 4))
plt.hist(lat.clip(0, 30), bins=50, color="steelblue", edgecolor="white")
plt.axvline(lat.quantile(0.99), color="red", linestyle="--", label=f"P99={lat.quantile(0.99):.1f}ms")
plt.xlabel("Latency (ms)")
plt.ylabel("Count")
plt.title("End-to-end pipeline latency distribution")
plt.legend()
plt.tight_layout()
plt.savefig("../data/processed/06_latency.png", dpi=120)
plt.show()

## 7. Per-class detection breakdown

In [ ]:
print("\n=== Detection by attack class ===")
for cls in ["sqli", "xss", "lfi", "other_attack"]:
    cls_df   = df_results[df_results["true_label"] == cls]
    detected = (cls_df["decision"] == "block").sum()
    total    = len(cls_df)
    print(f"  {cls:<15}: {detected}/{total}  ({detected/total*100:.1f}% detected)")

print("\n=== False positives (normal traffic blocked) ===")
normal_df = df_results[df_results["true_label"] == "normal"]
fp_count  = (normal_df["decision"] == "block").sum()
print(f"  {fp_count}/{len(normal_df)} normal requests blocked  "
      f"({fp_count/len(normal_df)*100:.2f}% FPR)")

## 8. Score distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, group) in zip(axes, [
    ("normal", df_results[df_results["true_label_id"] == 0]),
    ("attacks", df_results[df_results["true_label_id"] > 0]),
]):
    ax.hist(group["score"], bins=40, color="steelblue" if label=="normal" else "coral",
            edgecolor="white")
    ax.axvline(30, color="orange", linestyle="--", label="log threshold (30)")
    ax.axvline(70, color="red",    linestyle="--", label="block threshold (70)")
    ax.set_title(f"Threat score — {label}")
    ax.set_xlabel("Threat score")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../data/processed/06_score_distribution.png", dpi=120)
plt.show()

# Save full results table
df_results.to_csv(DATA_PROC / "06_pipeline_results.csv", index=False)
print("Results saved to data/processed/06_pipeline_results.csv")